КТ5, Лазарева А.К., 3П3.24

In [1]:
!pip install allure-pytest pydantic -q

Модели Pydantic

In [2]:
%%writefile models.py
from pydantic import BaseModel


class Category(BaseModel):
    id: int = 1
    name: str = "cats"


class Tag(BaseModel):
    id: int
    name: str


class Pet(BaseModel):
    id: int
    category: Category = Category()
    name: str
    photoUrls: list[str] = []
    tags: list[Tag] = []
    status: str = "available"


class Order(BaseModel):
    id: int
    petId: int
    quantity: int = 1
    shipDate: str | None = None
    status: str = "placed"
    complete: bool = False


class ApiResponse(BaseModel):
    code: int
    type: str
    message: str

Writing models.py


Базовый класс

In [3]:
%%writefile base_request.py
import json
import time

import allure
import requests

BASE_URL = "https://petstore.swagger.io/v2"


class BaseRequest:
    def __init__(self, base_url):
        self.base_url = base_url

    def _request(self, method, endpoint, data=None, expected_status=200):
        url = f"{self.base_url}/{endpoint}"
        with allure.step(f"{method} {url}"):
            for _ in range(5):
                response = requests.request(method, url, json=data)
                if response.status_code == expected_status:
                    break
                time.sleep(1)
            if data:
                allure.attach(json.dumps(data, ensure_ascii=False), "request", allure.attachment_type.JSON)
            allure.attach(response.text, "response", allure.attachment_type.JSON)
        return response

    def get(self, endpoint, expected_status=200):
        return self._request("GET", endpoint, expected_status=expected_status)

    def post(self, endpoint, data):
        return self._request("POST", endpoint, data)

    def put(self, endpoint, data):
        return self._request("PUT", endpoint, data)

    def delete(self, endpoint):
        return self._request("DELETE", endpoint)

Writing base_request.py


Pet

In [4]:
%%writefile test_pet.py
import allure
import pytest

from base_request import BASE_URL, BaseRequest
from models import ApiResponse, Pet

api = BaseRequest(BASE_URL)


@pytest.fixture
def pet():
    return Pet(id=332419, name="Муртузик")


@allure.feature("Pet")
@allure.title("Добавление питомца")
def test_post_pet(pet):
    response = api.post("pet", pet.model_dump())
    assert response.status_code == 200
    assert Pet.model_validate(response.json()) == pet


@allure.feature("Pet")
@allure.title("Получение питомца по id")
def test_get_pet(pet):
    response = api.get(f"pet/{pet.id}")
    assert response.status_code == 200
    result = Pet.model_validate(response.json())
    assert result.name == "Муртузик"
    assert result.category.name == "cats"


@allure.feature("Pet")
@allure.title("Изменение питомца")
def test_put_pet(pet):
    pet.name = "Кутузик"
    pet.status = "sold"
    response = api.put("pet", pet.model_dump())
    assert response.status_code == 200
    result = Pet.model_validate(response.json())
    assert result.name == "Кутузик"
    assert result.status == "sold"


@allure.feature("Pet")
@allure.title("Удаление питомца")
def test_delete_pet(pet):
    response = api.delete(f"pet/{pet.id}")
    assert response.status_code == 200
    assert ApiResponse.model_validate(response.json()).message == str(pet.id)
    response = api.get(f"pet/{pet.id}", expected_status=404)
    assert response.status_code == 404

Writing test_pet.py


In [5]:
!pytest test_pet.py --alluredir=allure-results --clean-alluredir -q

....                                                                     [100%]
4 passed in 24.02s


Store

In [6]:
%%writefile test_store.py
import allure
import pytest

from base_request import BASE_URL, BaseRequest
from models import ApiResponse, Order

api = BaseRequest(BASE_URL)


@pytest.fixture
def order():
    return Order(id=3324, petId=332419, quantity=2)


@allure.feature("Store")
@allure.title("Создание заказа")
def test_post_order(order):
    response = api.post("store/order", order.model_dump(exclude_none=True))
    assert response.status_code == 200
    result = Order.model_validate(response.json())
    assert result.id == order.id
    assert result.status == "placed"


@allure.feature("Store")
@allure.title("Получение заказа по id")
def test_get_order(order):
    response = api.get(f"store/order/{order.id}")
    assert response.status_code == 200
    result = Order.model_validate(response.json())
    assert result.petId == order.petId
    assert result.quantity == 2


@allure.feature("Store")
@allure.title("Получение остатков магазина")
def test_get_inventory():
    response = api.get("store/inventory")
    assert response.status_code == 200
    assert isinstance(response.json(), dict)


@allure.feature("Store")
@allure.title("Удаление заказа")
def test_delete_order(order):
    response = api.delete(f"store/order/{order.id}")
    assert response.status_code == 200
    assert ApiResponse.model_validate(response.json()).message == str(order.id)
    response = api.get(f"store/order/{order.id}", expected_status=404)
    assert response.status_code == 404

Writing test_store.py


In [7]:
!pytest test_store.py --alluredir=allure-results -q

....                                                                     [100%]
4 passed in 7.30s


Отчёт Allure

In [8]:
!npx allure awesome allure-results --single-file

the report successfully generated (1439ms)
